# Cobrinha ASCII
Execute a célula, clique no desenho. O jogo começa automaticamente. Use as setas ou botões.

Python exibe HTML; JavaScript controla o jogo no navegador. Não precisa instalar pacotes.

A cobrinha atravessa as bordas e o próprio corpo. O Tetris usa blocos ASCII, o Pong é contra o computador e o RPG pede uma chave para abrir a porta.

In [ ]:
from IPython.display import Javascript, display

# Execute esta célula inteira. Clique no desenho e use as setas.
# Python exibe a saída; JavaScript controla o jogo no navegador.
display(Javascript(r"""(() => {
// O painel fica na saída desta célula, sem iframe dentro de iframe.
const painel = document.createElement('div');
painel.style = 'background:#111;color:#a6ff99;padding:12px;max-width:100%;font:14px monospace';
painel.innerHTML = "<b>Cobrinha ASCII</b><p><button data-action=\"restart\">Reiniciar</button> <button data-action=\"pause\">Pausar / continuar</button></p><span role=\"status\"></span><pre tabindex=\"0\" aria-label=\"Jogo ASCII: use as setas\" style=\"font:14px/1.2 monospace;overflow:auto;outline:1px solid #486;padding:6px\"></pre><div><button data-key=\"ArrowUp\">Cima</button><button data-key=\"ArrowDown\">Baixo</button><button data-key=\"ArrowLeft\">Esquerda</button><button data-key=\"ArrowRight\">Direita</button></div>";
document.body.appendChild(painel);
const tela = painel.querySelector('pre'), aviso = painel.querySelector('[role="status"]');
let tempo = null, rodando = false, terminou = false;
function moldura(linhas) {
    const borda = '+' + '-'.repeat(linhas[0].length) + '+';
    return [borda, ...linhas.map(l => '|' + l + '|'), borda].join(String.fromCharCode(10));
}
// As coordenadas são [x, y]. As bordas se conectam pelo operador %.
const largura = 20, altura = 10;
const teclas = {ArrowUp:[0,-1], ArrowDown:[0,1], ArrowLeft:[-1,0], ArrowRight:[1,0]};
let cobra, direcao, comida, pontos, virou;
function sortearComida() {
    const livres = [];
    for (let y = 0; y < altura; y++) {
        for (let x = 0; x < largura; x++) {
            if (!cobra.some(p => p[0] === x && p[1] === y)) livres.push([x,y]);
        }
    }
    if (!livres.length) { terminou = true; return null; }
    return livres[Math.floor(Math.random() * livres.length)];
}
function reiniciar() {
    cobra = [[5,5],[4,5]];
    direcao = [1,0]; pontos = 0; virou = false;
    comida = sortearComida();
}
function comando(tecla) {
    const nova = teclas[tecla];
    if (!nova || virou) return;
    if (nova[0] === -direcao[0] && nova[1] === -direcao[1]) return;
    direcao = nova; virou = true;
}
function atualizar() {
    const x = (cobra[0][0] + direcao[0] + largura) % largura;
    const y = (cobra[0][1] + direcao[1] + altura) % altura;
    cobra.unshift([x,y]); virou = false;
    if (comida && x === comida[0] && y === comida[1]) {
        pontos += 10; comida = sortearComida();
    } else cobra.pop();
    // Modo livre: passar pelo próprio corpo não encerra a partida.
}
function desenhar() {
    const linhas = [];
    for (let y = 0; y < altura; y++) {
        let linha = '';
        for (let x = 0; x < largura; x++) {
            if (cobra[0][0] === x && cobra[0][1] === y) linha += '@';
            else if (cobra.some(p => p[0] === x && p[1] === y)) linha += 'o';
            else if (comida && comida[0] === x && comida[1] === y) linha += '*';
            else linha += ' ';
        }
        linhas.push(linha);
    }
    return moldura(linhas) + '\nPontos: ' + pontos + (terminou ? ' | VOCE VENCEU!' : '\nSetas: mover | bordas infinitas | sem colisao');
}

function parar() { clearInterval(tempo); tempo = null; rodando = false; }
function mostrar() {
    tela.textContent = desenhar();
    if (terminou) { parar(); aviso.textContent = 'Fim da partida. Use Reiniciar.'; }
}
function continuar() {
    if (terminou) return;
    parar(); rodando = true;
    aviso.textContent = 'Clique no desenho e use as setas, ou toque nos botoes.';
    tempo = setInterval(() => { if (!painel.isConnected) { parar(); return; } atualizar(); mostrar(); }, 200);
    tela.focus();
}
function iniciar() { parar(); terminou = false; reiniciar(); mostrar(); continuar(); }
function receber(tecla) { if (rodando && !terminou) { comando(tecla); mostrar(); } }
tela.onkeydown = e => {
    if (['ArrowUp','ArrowDown','ArrowLeft','ArrowRight',' '].includes(e.key)) {
        e.preventDefault(); receber(e.key);
    }
};
painel.querySelectorAll('[data-key]').forEach(b => b.onclick = () => { receber(b.dataset.key); tela.focus(); });
painel.querySelector('[data-action="restart"]').onclick = iniciar;
painel.querySelector('[data-action="pause"]').onclick = () => {
    if (rodando) { parar(); aviso.textContent = 'Pausado.'; } else continuar();
};
window.addEventListener('pagehide', parar, {once:true});
iniciar();
})();"""))